In [1]:
# ===== Cell 1: 用 Langchain 的 WebBaseLoader 爬取知乎文章 =====
# WebBaseLoader 是 langchain_community 提供的网页抓取工具
# 它用 requests 获取网页，用 BeautifulSoup 解析正文
# 知乎有反爬(403)，如果爬取失败或正文太短，自动回退到本地缓存文件
import os
from langchain_community.document_loaders import WebBaseLoader

url = 'https://zhuanlan.zhihu.com/p/333623112'

# 尝试用 Langchain 的 WebBaseLoader 爬取
docs = []
try:
    loader = WebBaseLoader(
        url,
        header_template={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        },
    )
    docs = loader.load()
    content_len = len(docs[0].page_content) if docs else 0
    print(f'WebBaseLoader 爬取到 {len(docs)} 篇文档, 正文 {content_len} 字符')
    if content_len < 100:
        print('正文太短，可能被反爬拦截，回退到本地缓存')
        docs = []
except Exception as e:
    print(f'WebBaseLoader 爬取失败: {e}')

# 回退到本地缓存（爬取失败或正文太短）
if not docs:
    cache_path = os.path.join(os.getcwd(), 'article_333623112.txt')
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8') as f:
            text = f.read()
        from langchain_core.documents import Document
        docs = [Document(page_content=text, metadata={'source': url})]
        print(f'已从本地缓存加载: {len(text)} 字符')
    else:
        print('缓存文件也不存在！')

print(f'\n最终文档数: {len(docs)}, 正文长度: {len(docs[0].page_content) if docs else 0}')


C:\Users\rehuanoo\AppData\Local\Temp\ipykernel_31680\4056326590.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


WebBaseLoader 爬取到 1 篇文档, 正文 37 字符
正文太短，可能被反爬拦截，回退到本地缓存
已从本地缓存加载: 5536 字符

最终文档数: 1, 正文长度: 5536


In [2]:
# ===== Cell 2: 加载本地 bge-large-zh 嵌入模型 =====
import os
import subprocess
import time
import torch
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings import Embeddings

# 自动创建 X: 盘（解决中文路径问题）
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists('X:\\'):
    subprocess.run(['subst', 'X:', project_root], check=False)

EMBED_PATH = r'X:\6.transformers\models\bge-large-zh-v1.5'


class BGEEmbeddings(Embeddings):
    '''本地 bge-large-zh 向量模型: CLS 池化 + L2 归一化, 输出 1024 维'''

    def __init__(self, model_path, device='cpu'):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path).to(device).eval()

    def _embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        with torch.no_grad():
            inputs = self.tokenizer(
                texts, padding=True, truncation=True,
                max_length=512, return_tensors='pt',
            ).to(self.device)
            outputs = self.model(**inputs)
            cls = outputs.last_hidden_state[:, 0]
            cls = torch.nn.functional.normalize(cls, p=2, dim=1)
        return cls.cpu().tolist()

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed(text)[0]


embeddings = BGEEmbeddings(EMBED_PATH, device='cpu')

t0 = time.time()
vec = embeddings.embed_query('测试向量')
print(f'嵌入模型就绪: 维度={len(vec)}, 耗时={time.time()-t0:.2f}s')


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6614.74it/s]


嵌入模型就绪: 维度=1024, 耗时=2.17s


In [3]:
# ===== Cell 3: 文本分块 + 构建 FAISS 向量库 =====
# 用 Langchain 的 RecursiveCharacterTextSplitter 对文章做分块
# 用 Langchain 的 FAISS 构建向量库
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 1) 文本分块：每块最大 300 字符，重叠 50 字符
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n', '。', '；', '，', ' '],
)
chunks = splitter.split_documents(docs)
print(f'分块后文档数: {len(chunks)}')

# 2) 构建 FAISS 向量库（Langchain 封装，底层调用 embeddings 做向量化）
t0 = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'FAISS 向量库构建完成: {len(chunks)} 个块, 耗时 {time.time()-t0:.1f}s')


分块后文档数: 23
FAISS 向量库构建完成: 23 个块, 耗时 5.6s


In [4]:
# ===== Cell 4: 接入付费模型 + 组装 RAG 问答链 =====
# 用 langchain_openai 的 ChatOpenAI 接入商汤 SenseNova 付费模型
# 兼容 OpenAI 接口格式，只需改 base_url 和 api_key
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 商汤 SenseNova 付费模型（与 01.Langchain.ipynb 中的配置一致）
llm = ChatOpenAI(
    model='sensenova-6.8-flash-lite',
    base_url='https://token.sensenova.cn/v1',
    api_key=os.getenv('SENSENOVA_API_KEY', ''),
    temperature=0.3,   # 知识库问答用低温度，答案更稳定
)

# RAG 提示词：要求模型严格依据检索到的资料回答
template = '''请根据以下参考资料回答用户问题。
如果资料中没有相关内容，请直接回答"根据现有资料无法回答该问题"。

参考资料：
{context}

用户问题：{question}'''

prompt = ChatPromptTemplate.from_template(template)

# 用 Langchain 的 as_retriever 将 FAISS 向量库转为检索器
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

# LCEL 链：检索 -> 格式化上下文 -> 提示词 -> 大模型 -> 字符串输出
def format_docs(docs):
    return '\n\n'.join(f'[资料{i+1}] {d.page_content}' for i, d in enumerate(docs))

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG 问答链组装完成，模型: sensenova-6.8-flash-lite')
print('知识库: 知乎文章"50个科学小知识"')


RAG 问答链组装完成，模型: sensenova-6.8-flash-lite
知识库: 知乎文章"50个科学小知识"


In [5]:
# ===== Cell 5: 测试 RAG 问答 =====
import time

questions = [
    '为什么向日葵总是向着太阳？',
    '为什么星星会一闪一闪的？',
    '为什么人老了头发会变白？',
    '为什么海水是蓝色的？',
    '婴儿出生时为什么会哭？',
]

for q in questions:
    t0 = time.time()
    answer = rag_chain.invoke(q)
    print(f'\n问: {q}')
    print(f'答: {answer}')
    print(f'（耗时 {time.time()-t0:.1f}s）')



问: 为什么向日葵总是向着太阳？
答: 

根据参考资料，向日葵总是向着太阳的原因如下：

向日葵的茎部含有一种奇妙的植物生长素。这种生长素非常怕光，一遇光线照射，它就会到背光的一面去，同时它还刺激背光一面的细胞迅速繁殖。因此，背光的一面比向光的一面生长得更快，使向日葵产生了向光性弯曲，从而总是朝着太阳的方向生长。
（耗时 5.3s）

问: 为什么星星会一闪一闪的？
答: 

根据参考资料，星星一闪一闪并不是因为星星本身的光度出现变化，而是与大气的遮挡有关。大气隔在我们与星星之间，当星光通过大气层时，会受到大气的密度和厚薄影响。大气不是绝对的透明，它的透明度会根据密度的不同而产生变化。所以我们在地面透过它来看星星，就会看到星星好像在闪动的样子了。
（耗时 2.8s）

问: 为什么人老了头发会变白？
答: 

根据资料，人老了头发会变白的原因如下：

我们的头发中有一种叫「黑色素」的物质，黑色素愈多头发的颜色便愈黑。而黑色素少的话，头发便会发黄或变白。人类到了老年时，身体的各种机能会逐渐衰退，色素的形成亦会愈来愈少，所以头发也会渐渐变白。
（耗时 21.9s）

问: 为什么海水是蓝色的？
答: 

根据参考资料，海水本身与我们日常接触到的水没有大分别，也是透明无色的。我们所看到的颜色其实是海水对光的吸收能力而产生的现象。只有绿光能被海水吸收并反射出来；当海水更深时，绿光也被吸收，海水看上去便成了蓝色。
（耗时 44.5s）

问: 婴儿出生时为什么会哭？
答: 

根据参考资料，婴儿刚出生时呱呱大哭，并不是因为他们感到不开心，而是他们正在大口大口地呼吸着第一口空气。当婴儿离开妈妈身体出生时，他们吸进的第一口空气会冲到喉部去，这会猛烈地冲击声带，令声带震动，然后发出类似哭叫的声音。
（耗时 25.5s）


In [6]:
rag_chain.invoke('讲讲螳螂')

'\n\n根据现有资料，关于螳螂的信息非常有限，仅有一条记录：\n\n**螳螂被称为"捕虫神刀手"。**\n\n资料中并未提供更多关于螳螂的详细描述，如需了解更多关于螳螂的知识，建议查阅其他资料。'